Opening Transmission Record/clean audio loading + sampling rate selection

In [6]:
from IPython.display import Audio
from scipy.io.wavfile import write, read
import numpy as np
import sounddevice as sd
import scipy.signal as signal
import matplotlib.pyplot as plt
from tune import *



file_path = "clean_transmission_sound.wav"
fs, audio_data = read(file_path)
print(f"Sampling rate: {fs} Hz")
#print(f"Audio data shape: {audio_data.shape}")
Audio(audio_data, rate=fs)


Sampling rate: 44100 Hz


In [ ]:

# We add the sound to  be detected to the background 

In [7]:
def prewarp_frequency(f, fs):
    """
    Pre-warp the digital frequency f and convert it to angular frequency w (rad/s).
    Returns pre-warped angular frequency (wc) in rad/s.
    """
    if fs <= 0:
        raise ValueError("Sampling frequency must be positive.")
    if not (0 < f < fs / 2):
        raise ValueError("Frequency must be between 0 and Nyquist frequency.")

    Ts = 1.0 / fs  # Sampling period
    w = 2 * np.pi * f  # Digital angular frequency in rad/s

    # Pre-warping 
    pre_warped_w = 2/Ts * np.tan(w * Ts / 2)

    return pre_warped_w

### Gradual interference entering an anomaly zone (narrowband noise 6-8 kHz)

In [ ]:
passband_1 = 6e3
passband_2 = 8e3
order = 4  # filter order
b, a = signal.iirfilter(Wn=1.0, N=order, analog=True, ftype='butter', btype='low')  # Design an analog lowpass Butterworth filter with cutoff frequency of 1 rad/s

#print(f"Numerator Coefficients lowpass filter: {b}")
#print(f"Denominator Coefficients lowpass filter: {a}")

omega_passband1 = prewarp_frequency(passband_1, fs)
omega_passband2 = prewarp_frequency(passband_2, fs)

omega_passband_center = np.sqrt(omega_passband1 * omega_passband2) # we use the geometric mean to find the center frequency of the passband, alternative calcutlate f_center and then prearp
omega_passband_width = omega_passband2 - omega_passband1

# We apply the prewarping to the passband frequencies before converting the lowpass filter to a bandpass filter. 
b,a = signal.lp2bp(b, a, omega_passband_center, omega_passband_width)  # convert to bandpass filter 
# note the order of the filter is doubled when converting from lowpass to bandpass, so we have an 8th order bandpass filter.

b,a = signal.bilinear(b, a, fs=fs)


In [ ]:
# Gradual interference entering an anomaly zone (narrowband noise 6-8 kHz)
signal = signal.lfilter(b, a, audio_data)



Sudden signal degradation near a metallic moon (broadband noise + drop-outs) 

In [ ]:
length_s = 2  # length of the audio signal in seconds
noise = np.random.normal(-1, 1, length_s * fs) # Generate white noise


b,a = signal.iirfilter(4, [6000, 8000], fs=fs, btype='band', ftype='butter')
filtered_noise = signal.lfilter(b, a, noise)

# We now add the filtered noise to the original audio signal to create a new audio signal with interference


### Temporary blackout crossing behind the moon ( 2 seconds full signal loss) 

### Gradual recovery emerging from the moon shadow

### Return to normal communication at the end